# EXP-01: EDA — Phân tích dữ liệu ViHSD

**TV1 — Data Analysis & Traditional ML**  
Dataset: UIT-ViHSD (`ura-hcmut/UIT-ViHSD`)  
Nhãn: CLEAN=0, OFFENSIVE=1, HATE=2

> **Trạng thái:** ĐÃ THỰC HIỆN — dữ liệu UIT-ViHSD đã tải (`data/processed/vihsd.jsonl`, train 24.048 / val 2.672); EDA chạy xong qua 5 bước, figure lưu trong `notebooks/`.

## Mục tiêu
1. Phân phối nhãn và class imbalance
2. Thống kê độ dài văn bản
3. Từ phổ biến theo lớp
4. Phát hiện patterns: URL, emoji, teencode, HTML
5. Exact duplicate & cross-split overlap check
6. Nhận xét và quyết định preprocessing


In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.eda.eda_utils import (
    load_jsonl_to_df,
    plot_label_distribution,
    print_label_stats,
    compute_text_lengths,
    plot_text_length_distribution,
    print_length_stats,
    plot_top_words,
    detect_patterns,
    print_pattern_stats,
    check_duplicates,
)

DATA_PATH = ROOT / 'data' / 'processed' / 'vihsd.jsonl'
print(f'Data path: {DATA_PATH}')
print(f'Exists: {DATA_PATH.exists()}')


In [ ]:
if not DATA_PATH.exists():
    print('CHƯA THỰC HIỆN: Dữ liệu chưa được tải.')
    print('Chạy: python -m src.data.download_dataset')
else:
    df = load_jsonl_to_df(DATA_PATH)
    print(f'Loaded {len(df):,} records')
    print(df.head())
    print(df.dtypes)


## 1. Phân phối nhãn

In [ ]:
if 'df' in dir():
    print_label_stats(df)
    fig = plot_label_distribution(df)
    fig.savefig(ROOT / 'notebooks' / 'eda_label_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('CHƯA THỰC HIỆN')


## 2. Thống kê độ dài văn bản

In [ ]:
if 'df' in dir():
    df = compute_text_lengths(df)
    print_length_stats(df)
    fig = plot_text_length_distribution(df)
    fig.savefig(ROOT / 'notebooks' / 'eda_text_lengths.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('CHƯA THỰC HIỆN')


## 3. Từ phổ biến theo lớp

In [ ]:
if 'df' in dir():
    fig = plot_top_words(df, top_n=20, split='train')
    fig.savefig(ROOT / 'notebooks' / 'eda_top_words.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('CHƯA THỰC HIỆN')


## 4. Phát hiện patterns đặc thù

In [ ]:
if 'df' in dir():
    df = detect_patterns(df)
    print_pattern_stats(df)
else:
    print('CHƯA THỰC HIỆN')


## 5. Duplicate & Cross-split overlap check

In [ ]:
if 'df' in dir():
    dup_report = check_duplicates(df)
    import json
    print(json.dumps(dup_report, ensure_ascii=False, indent=2, default=str))
else:
    print('CHƯA THỰC HIỆN')


## 6. Nhận xét và quyết định preprocessing

> **Kết quả EDA thực tế:**
>
> - Class imbalance: train CLEAN 82,7% / OFFENSIVE 6,7% / HATE 10,6% → cần class_weight hoặc resampling, không bỏ qua OFFENSIVE/HATE.
> - Độ dài trung bình: 11,5 từ (median 8), max 1.701 → có outlier rất dài; cân nhắc truncation ở sequence length hợp lý (ví dụ 64–128).
> - Tỷ lệ URL/emoji/teencode: URL <0,1%, HTML ~0%; emoji 11% CLEAN / 8,5% OFFENSIVE / 5,3% HATE; short tokens 77–89% → giữ emoji, bỏ HTML/URL ít có tác dụng, xem xét teencode normalization.
> - Có duplicate/overlap không: 3.586 exact duplicates; 318 text trùng giữa train và val (leakage) → phải xử lý deduplication và đảm bảo không overlap train/val trước khi training.
> - Quyết định preprocessing variant A/B/C: ưu tiên variant teencode normalization + class_weight (B/C) để cải thiện OFFENSIVE/HATE recall (baseline A OFFENSIVE F1 chỉ 0,21).

